<a href="https://colab.research.google.com/github/EliasRH1/prv/blob/main/eliastupapi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!sudo apt-get update && sudo apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,044 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [78.0 kB]
Get:7 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:8 http://archive.ubuntu.com/ubuntu noble InRelease
Get:9 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.5 MB]
Hit:13 https://ppa.launchpadcontent.net/graphics-dri

In [4]:
!nohup ollama serve > ollama.log 2>&1 &
import time
time.sleep(5)
!cat ollama.log | grep "API"
print("✅ Servidor listo para usarse con GPU.")

✅ Servidor listo para usarse con GPU.


In [5]:
!ollama pull llama3.1

In [7]:
!pip install gradio

In [13]:
import subprocess
import time
import os
import ollama
import gradio as gr

print("1. Iniciando servidor Ollama...")
!pkill ollama
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("✅ Servidor en línea.")

def responder_chat(mensaje, historial):
    try:
        mensajes_ollama = []

        # Procesar historial de forma segura extrayendo siempre texto plano
        for turno in historial:
            # Gradio moderno envía diccionarios o listas de objetos
            if isinstance(turno, dict):
                role = "user" if turno.get("role") == "user" else "assistant"
                content = turno.get("content", "")
                if isinstance(content, list):
                    content = " ".join([str(item.get("text", "")) for item in content if isinstance(item, dict)])
                if content:
                    mensajes_ollama.append({'role': role, 'content': str(content)})
            elif isinstance(turno, (list, tuple)) and len(turno) == 2:
                hum, bot = turno
                if hum: mensajes_ollama.append({'role': 'user', 'content': str(hum)})
                if bot: mensajes_ollama.append({'role': 'assistant', 'content': str(bot)})

        # Agregar el mensaje actual del usuario
        mensajes_ollama.append({'role': 'user', 'content': str(mensaje)})

        # Llamada limpia a Ollama
        respuesta = ollama.chat(
            model='llama3.1',
            messages=mensajes_ollama
        )
        return respuesta['message']['content']

    except Exception as e:
        return f"⚠️ Error de conexión con Ollama: {e}"

print("2. Lanzando interfaz web...")
demo = gr.ChatInterface(
    fn=responder_chat,
    title="🤖 Llama 3.1 (Ollama + Colab)",
    description="Tu interfaz privada conectada a la GPU T4."
)

demo.launch(share=True, debug=False, inline=True)

1. Iniciando servidor Ollama...
✅ Servidor en línea.
2. Lanzando interfaz web...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b439fed45458f7e2e0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
